In [21]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [22]:
with open("data/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [23]:
context_length = 8

x = enc_text[:context_length]
y = enc_text[1:context_length + 1]

print("Input tokens:", x)
print("Target tokens:   ", y)

Input tokens: [40, 367, 2885, 1464, 1807, 3619, 402, 271]
Target tokens:    [367, 2885, 1464, 1807, 3619, 402, 271, 10899]


In [24]:
import torch

print(torch.__version__)

2.8.0


In [25]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

Using device: mps


In [26]:
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, enc_text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        
        token_ids = tokenizer.encode(enc_text, allowed_special={"<|endoftext|>"})
        
        for i in range(0, len(token_ids) - max_length, stride):
            input_seq = token_ids[i:i + max_length]
            target_seq = token_ids[i + 1:i + max_length + 1]
            
            self.input_ids.append(torch.tensor(input_seq, dtype=torch.long))
            self.target_ids.append(torch.tensor(target_seq, dtype=torch.long))
            
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


In [27]:
def create_dataloader(txt, batch_size = 2, max_length = 256,
                      stride = 128, shuffle = True, drop_last = True,
                      num_workers = 0):
    
    tokenizer = tiktoken.get_encoding("gpt2")
    
    dataset = GPTDataset(txt, tokenizer, max_length, stride)
    
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                            drop_last=drop_last, num_workers=num_workers)
    
    return dataloader



In [28]:
dataloaders = create_dataloader(raw_text, batch_size=1, max_length=4, stride=1)

dataloader_iter = iter(dataloaders)
first_batch = next(dataloader_iter)
print("Input batch:", first_batch)

Input batch: [tensor([[1068,  284, 1577,  257]]), tensor([[  284,  1577,   257, 23844]])]


In [29]:
second_batch = next(dataloader_iter)
print("Input batch:", second_batch)

Input batch: [tensor([[ 383, 8631, 3872,  373]]), tensor([[8631, 3872,  373,   11]])]


In [30]:
dataloaders = create_dataloader(raw_text, batch_size=8, max_length=4, stride=4)

dataloader_iter = iter(dataloaders)
first_batch = next(dataloader_iter)
print("Input batch:", first_batch)

Input batch: [tensor([[  262,  4269,    11, 22211],
        [ 2236,   407,   804,  2402],
        [  503,   465,  7721,   257],
        [ 9813,   286,  4190,  7121],
        [  198,   198,    40, 27846],
        [  465,  1182,   284,   804],
        [ 1466,   550,   925,   683],
        [  257, 14093,   526,   198]]), tensor([[ 4269,    11, 22211,   262],
        [  407,   804,  2402,   663],
        [  465,  7721,   257,  1310],
        [  286,  4190,  7121,   736],
        [  198,    40, 27846,   706],
        [ 1182,   284,   804,   510],
        [  550,   925,   683,   438],
        [14093,   526,   198,   198]])]


In [31]:
vocab_size = tokenizer.n_vocab
output_dim = 256

torch.manual_seed(42)
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [32]:
token_embedding_layer.weight.shape

torch.Size([50257, 256])

In [33]:
token_embedding = token_embedding_layer(first_batch[0])
print("Token embedding shape:", token_embedding.shape)

Token embedding shape: torch.Size([8, 4, 256])


In [34]:
context_length = 4
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

pos_embedding = pos_embedding_layer(torch.arange(context_length))

In [38]:
pos_embedding.shape

torch.Size([4, 256])

In [36]:
pos_embedding

tensor([[-0.5226,  0.7391, -0.6129,  ...,  1.4309,  0.5757, -0.2500],
        [-1.3638,  0.5608,  0.8688,  ..., -0.7332,  0.1195, -1.0887],
        [-1.1647,  2.5871,  0.5725,  ...,  1.8961, -0.2037, -0.7033],
        [-0.3663,  0.5819,  0.3062,  ..., -0.3303, -0.3610,  0.6517]],
       grad_fn=<EmbeddingBackward0>)

In [37]:
input_embedding = token_embedding + pos_embedding
print("Input embedding shape:", input_embedding.shape)

Input embedding shape: torch.Size([8, 4, 256])
